In [38]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [39]:
train_df = pd.read_csv("rossmann-store-sales/train.csv")
store_df = pd.read_csv("rossmann-store-sales/store.csv")

C:\Users\zand\AppData\Local\Temp\ipykernel_12236\3545578414.py:1: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  train_df = pd.read_csv("rossmann-store-sales/train.csv")


In [40]:
train_df.shape

(1017209, 9)

In [41]:
train_df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday'],
      dtype='object')

In [42]:
train_df['Date'] = pd.to_datetime(train_df["Date"])
train_df['year'] = train_df['Date'].dt.year 
train_df['month'] = train_df['Date'].dt.month
train_df['week'] = train_df['Date'].dt.isocalendar().week.astype(int)
train_df['day'] = train_df['Date'].dt.day

train_df.drop(columns= 'Date', inplace=True)
train_df.head()

,Store,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,year,month,week,day
0,1,5,5263,555,1,1,0,1,2015,7,31,31
1,2,5,6064,625,1,1,0,1,2015,7,31,31
2,3,5,8314,821,1,1,0,1,2015,7,31,31
3,4,5,13995,1498,1,1,0,1,2015,7,31,31
4,5,5,4822,559,1,1,0,1,2015,7,31,31


In [43]:
target = ["Sales"]
num_cols = ['Customers', 'day', 'month', 'year', 'week']
cat_cols = ['Store', 'DayOfWeek', 'Open', 'Promo', 'StateHoliday', 'SchoolHoliday']


In [44]:
for col in cat_cols:
    print(col, train_df[col].nunique())

Store 1115
DayOfWeek 7
Open 2
Promo 2
StateHoliday 5
SchoolHoliday 2


In [45]:
train_df["Date"] = pd.to_datetime(train_df[["year", "month", "day"]])
train_df = train_df.sort_values("Date").reset_index(drop=True)

In [46]:
train_df = train_df.sort_values(by=["Store", "Date"]).reset_index(drop=True)

In [47]:
for lag in [1, 7, 28]:
    train_df[f"sales_lag_{lag}"] = (
        train_df.groupby("Store")["Sales"].shift(lag)
    )
    

In [48]:
for window in [7, 28]:
    train_df[f"sales_std_{window}"] = (
        train_df
        .groupby("Store")["Sales"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

In [49]:
train_df.head()

,Store,DayOfWeek,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,year,month,week,day,Date,sales_lag_1,sales_lag_7,sales_lag_28,sales_std_7,sales_std_28
0,1,2,0,0,0,0,a,1,2013,1,1,1,2013-01-01,NaN,NaN,NaN,NaN,NaN
1,1,3,5530,668,1,0,0,1,2013,1,1,2,2013-01-02,0.0,NaN,NaN,NaN,NaN
2,1,4,4327,578,1,0,0,1,2013,1,1,3,2013-01-03,5530.0,NaN,NaN,NaN,NaN
3,1,5,4486,619,1,0,0,1,2013,1,1,4,2013-01-04,4327.0,NaN,NaN,NaN,NaN
4,1,6,4997,635,1,0,0,1,2013,1,1,5,2013-01-05,4486.0,NaN,NaN,NaN,NaN


In [50]:
train_df = train_df.dropna().reset_index(drop=True)

In [51]:
train_df.dtypes

Store                     int64
DayOfWeek                 int64
Sales                     int64
Customers                 int64
Open                      int64
Promo                     int64
StateHoliday             object
SchoolHoliday             int64
year                      int32
month                     int32
week                      int64
day                       int32
Date             datetime64[ns]
sales_lag_1             float64
sales_lag_7             float64
sales_lag_28            float64
sales_std_7             float64
sales_std_28            float64
dtype: object

In [52]:
train_df.drop(columns= 'Date', inplace=True)

In [53]:
train_df.drop(columns= 'Customers', inplace=True)

In [54]:
print(train_df["StateHoliday"].unique())

['0' 'b' 'a' 'c' 0]


In [55]:
train_df["StateHoliday"] = (
    train_df["StateHoliday"]
    .fillna("missing")
    .astype(str)
)

In [56]:
print(train_df["StateHoliday"].unique())

['0' 'b' 'a' 'c']


In [57]:
split_index = int(len(train_df) * 0.8)

train_data = train_df.iloc[:split_index]
test_data = train_df.iloc[split_index:]




In [58]:
train_data.columns.to_list()

['Store',
 'DayOfWeek',
 'Sales',
 'Open',
 'Promo',
 'StateHoliday',
 'SchoolHoliday',
 'year',
 'month',
 'week',
 'day',
 'sales_lag_1',
 'sales_lag_7',
 'sales_lag_28',
 'sales_std_7',
 'sales_std_28']

In [59]:
feature_cols = [
    'Store',
 'DayOfWeek',
 'Sales',
 'Open',
 'Promo',
 'StateHoliday',
 'SchoolHoliday',
 'year',
 'month',
 'week',
 'day',
 'sales_lag_1',
 'sales_lag_7',
 'sales_lag_28',
 'sales_std_7',
 'sales_std_28'
 ]

target =["Sales"]

In [60]:
X_train = train_data[feature_cols]
y_train = train_data[target]

X_test = test_data[feature_cols]
y_test = test_data[target]

print(X_train.shape)
print(y_train.shape)

print(X_test.shape)
print(y_test.shape)



(788791, 16)
(788791, 1)
(197198, 16)
(197198, 1)


In [61]:
y_train.head()

,Sales
0,3725
1,4601
2,4709
3,5633
4,5970


In [62]:
X_train.head()

,Store,DayOfWeek,Sales,Open,Promo,StateHoliday,SchoolHoliday,year,month,week,day,sales_lag_1,sales_lag_7,sales_lag_28,sales_std_7,sales_std_28
0,1,2,3725,1,0,0,0,2013,1,5,29,4055.0,5720.0,0.0,2080.017823,2075.631146
1,1,3,4601,1,0,0,0,2013,1,5,30,3725.0,5578.0,5530.0,2026.274696,1914.845455
2,1,4,4709,1,0,0,0,2013,1,5,31,4601.0,5195.0,4327.0,1951.681400,1899.913267
3,1,5,5633,1,0,0,0,2013,2,5,1,4709.0,5586.0,4486.0,1914.889329,1902.071860
4,1,6,5970,1,0,0,0,2013,2,5,2,5633.0,5598.0,4997.0,1921.288841,1919.949818


In [63]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder


In [64]:
preprocesor = ColumnTransformer(
    transformers=[
        (
            "cat", OneHotEncoder(handle_unknown="ignore"), ["StateHoliday"])
            ],
        remainder="passthrough"
)

In [65]:
X_train_processed = preprocesor.fit_transform(X_train)
X_test_processed = preprocesor.transform(X_test)

In [66]:
from xgboost import XGBRegressor



model = XGBRegressor(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    n_jobs=-1,
    random_state=42
)

model.fit(
    X_train_processed,
    y_train
)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [67]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

pred = model.predict(X_test_processed)

mae = mean_absolute_error(y_test, pred)

rmse = np.sqrt(
    mean_squared_error(y_test, pred)
)

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 19.598037719726562
RMSE: 116.22369073929161
